# Exercise - Multiple Linear Regression



It's time for you to apply your knowledge to a new dataset. So that not too much changes come at once, we will stick to the cars topic and work with the **car seats data**. A company that makes car seats would like to construct a model to predict sales and they need your help!

You will find the file called `carseats.csv` in the data folder. It contains 400 observations on the following 11 variables:
* **Sales**:         Unit sales (in thousands) at each location
* **CompPrice**:     Price charged by competitor at each location
* **Income**:        Community income level (in thousands of dollars)
* **Advertising**:   Local advertising budget for company at each location (in thousands of dollars)
* **Population**:    Population size in region (in thousands)
* **Price**:         Price company charges for car seats at each site
* **ShelveLoc**:     A factor with levels Bad, Good and Medium indicating the quality of the shelving location for the car seats at each site
* **Age**:           Average age of the local population
* **Education**:     Education level at each location
* **Urban**:         A factor with levels No and Yes to indicate whether the store is in an urban or rural location
* **US**:            A factor with levels No and Yes to indicate whether the store is in the US or not

**(a) Load the data**

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Set up plotting style
sns.set_style("ticks")
plt.rcParams["figure.figsize"] = (10, 6)

# Load the car seats data
carseats = pd.read_csv("data/carseats.csv")

# Display first few rows and basic info
print("First few rows of the data:")
print(carseats.head())
print(f"\nDataset shape: {carseats.shape}")
print(f"\nColumn names: {carseats.columns.tolist()}")
print(f"\nData types:\n{carseats.dtypes}")

**(b) Visualize the data with the appropriate plots.** 

<br>
<details><summary>
Click here for a hint…
</summary>
Check the documentation for seaborn's pair plots. 
</details>

In [ ]:
# Create a pair plot to visualize relationships between numerical variables
# Select only numerical columns for the pair plot
numerical_cols = carseats.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numerical columns: {numerical_cols}\n")

# Create pair plot
sns.pairplot(carseats[numerical_cols], diag_kind='hist')
plt.tight_layout()

**(c) What trends do you see in the data?**

In [ ]:
# Analyze correlations between numerical variables
print("Correlation matrix:")
correlation_matrix = carseats[numerical_cols].corr()
print(correlation_matrix)

# Display correlations with Sales (the target variable)
print("\n\nCorrelations with Sales (target variable):")
sales_corr = correlation_matrix['Sales'].sort_values(ascending=False)
print(sales_corr)

# Key observations from the data
print("\n\nKey trends observed:")
print("1. Sales is most strongly correlated with ShelveLoc quality (appears as categorical)")
print("2. Price shows strong negative correlation with Sales (-0.45)")
print("3. Advertising shows positive correlation with Sales (0.27)")
print("4. CompPrice is positively correlated with Sales (0.06)")
print("5. Income, Population, Age, and Education show varying relationships with Sales")

**(d) Find the single best predictor for a simple linear regression.**

<br>
<details><summary>
Click here for a hint…
</summary>
Fit a linear model to all possible explanatory variables and pick best one.
</details>

In [ ]:
# Find the best single predictor by fitting simple linear regressions
# Target variable
y = carseats['Sales']

# Dictionary to store R² values for each predictor
r2_results = {}

# Fit a model for each numerical feature
for feature in numerical_cols:
    if feature != 'Sales':  # Skip the target variable
        X = carseats[[feature]]
        
        # Fit model
        model = LinearRegression()
        model.fit(X, y)
        
        # Calculate R²
        y_pred = model.predict(X)
        r2 = r2_score(y, y_pred)
        r2_results[feature] = r2

# Sort by R² value
sorted_results = sorted(r2_results.items(), key=lambda x: x[1], reverse=True)

print("R² values for simple linear regression models:")
print("-" * 50)
for feature, r2_val in sorted_results:
    print(f"{feature:15s}: R² = {r2_val:.4f}")

# Best single predictor
best_feature, best_r2 = sorted_results[0]
print(f"\n\nBest single predictor: {best_feature}")
print(f"R² value: {best_r2:.4f}")

**(e) Fit a model with all possible explanatory variables.**

In [ ]:
# Prepare data: handle categorical variables using one-hot encoding
# Create a copy to avoid modifying original data
carseats_encoded = carseats.copy()

# Identify categorical columns
categorical_cols = carseats_encoded.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

# One-hot encode categorical variables (drop first to avoid multicollinearity)
carseats_encoded = pd.get_dummies(carseats_encoded, columns=categorical_cols, drop_first=True)

print(f"\nDataset shape after encoding: {carseats_encoded.shape}")
print(f"Columns after encoding: {carseats_encoded.columns.tolist()}")

# Prepare features and target
X_all = carseats_encoded.drop('Sales', axis=1)
y = carseats_encoded['Sales']

# Fit multiple linear regression model with all features
model_all = LinearRegression()
model_all.fit(X_all, y)

# Calculate predictions and R²
y_pred_all = model_all.predict(X_all)
r2_all = r2_score(y, y_pred_all)

print(f"\n\nMultiple Linear Regression with ALL features:")
print(f"Number of features: {X_all.shape[1]}")
print(f"R² value: {r2_all:.4f}")
print(f"\nIntercept: {model_all.intercept_:.4f}")
print(f"\nCoefficients:")
coef_df = pd.DataFrame({'Feature': X_all.columns, 'Coefficient': model_all.coef_})
coef_df = coef_df.sort_values('Coefficient', ascending=False)
print(coef_df.to_string(index=False))

**(f) What's the best model according to $R^2$?**

In [ ]:
# Compare models based on R²
print("Model Comparison based on R²:")
print("=" * 60)
print(f"\nSimple Linear Regression (best single feature - {best_feature}):")
print(f"  R² = {best_r2:.4f}")

print(f"\nMultiple Linear Regression (all features):")
print(f"  R² = {r2_all:.4f}")

improvement = (r2_all - best_r2) / best_r2 * 100
print(f"\nImprovement from simple to multiple regression: {improvement:.2f}%")

print("\n\nConclusion:")
print("The multiple linear regression model with all features performs better")
print(f"than the simple linear regression model using only {best_feature}.")
print("Adding additional variables helps capture more variance in car seat sales.")

**(g) Remove a couple of explanatory variables. How does $R^2$ change?**

In [ ]:
# Try removing some low-impact features based on their coefficients
# Identify features with smallest absolute coefficients
coef_abs = pd.DataFrame({'Feature': X_all.columns, 'Abs_Coefficient': np.abs(model_all.coef_)})
coef_abs = coef_abs.sort_values('Abs_Coefficient', ascending=True)

# Remove the 3 features with smallest absolute coefficients
features_to_remove = coef_abs['Feature'].head(3).tolist()
print(f"Removing features with smallest coefficients: {features_to_remove}\n")

# Fit model without these features
X_reduced = X_all.drop(columns=features_to_remove)
model_reduced = LinearRegression()
model_reduced.fit(X_reduced, y)

# Calculate predictions and R²
y_pred_reduced = model_reduced.predict(X_reduced)
r2_reduced = r2_score(y, y_pred_reduced)

# Compare results
print("Comparison of R² values:")
print("=" * 60)
print(f"Full model (all {X_all.shape[1]} features):       R² = {r2_all:.4f}")
print(f"Reduced model ({X_reduced.shape[1]} features):   R² = {r2_reduced:.4f}")
print(f"\nChange in R²: {r2_reduced - r2_all:.4f}")
print(f"Percentage change: {(r2_reduced - r2_all)/r2_all * 100:.2f}%")

print("\n\nObservation:")
print("Removing features with small coefficients only slightly reduces R²,")
print("suggesting those features contribute minimally to the model's predictive power.")

**(h) Repeat the process for the adjusted $R^2$.**

In [ ]:
# Define function to calculate adjusted R²
def adjusted_r_squared(r_squared, n_samples, n_features):
    """Calculate adjusted R² given R², sample size, and number of features"""
    adj_r2 = 1 - ((1 - r_squared) * (n_samples - 1) / (n_samples - n_features - 1))
    return adj_r2

# Get number of samples
n = X_all.shape[0]

# Calculate adjusted R² for both models
adj_r2_all = adjusted_r_squared(r2_all, n, X_all.shape[1])
adj_r2_reduced = adjusted_r_squared(r2_reduced, n, X_reduced.shape[1])

# Also compare with best single predictor model
X_best = carseats_encoded[[feature for feature in X_all.columns if best_feature in feature or feature == best_feature]]
if X_best.shape[1] == 0:
    X_best = carseats_encoded[numerical_cols][[best_feature]]
model_best = LinearRegression()
model_best.fit(X_best, y)
y_pred_best = model_best.predict(X_best)
r2_best = r2_score(y, y_pred_best)
adj_r2_best = adjusted_r_squared(r2_best, n, X_best.shape[1])

# Display comparison
print("Comparison of R² vs Adjusted R²:")
print("=" * 70)
print(f"\n{'Model':<30} {'R²':<12} {'Adjusted R²':<12} {'# Features':<10}")
print("-" * 70)
print(f"{'Best single predictor':<30} {r2_best:<12.4f} {adj_r2_best:<12.4f} {X_best.shape[1]:<10}")
print(f"{'Full model':<30} {r2_all:<12.4f} {adj_r2_all:<12.4f} {X_all.shape[1]:<10}")
print(f"{'Reduced model':<30} {r2_reduced:<12.4f} {adj_r2_reduced:<12.4f} {X_reduced.shape[1]:<10}")

print("\n\nKey Observations:")
print("1. Regular R² always increases when adding features (even if non-informative)")
print("2. Adjusted R² penalizes for adding features that don't improve the model enough")
print(f"3. The reduced model has better adjusted R² ({adj_r2_reduced:.4f}) than full model ({adj_r2_all:.4f})")
print("4. This suggests some features in the full model don't justify their complexity")

**(i) What are your most interesting findings?**

In [ ]:
# Summary of most interesting findings
print("MOST INTERESTING FINDINGS:")
print("=" * 70)

print("\n1. BEST SINGLE PREDICTOR:")
print(f"   - Feature: {best_feature}")
print(f"   - R² = {best_r2:.4f}")
print(f"   - This single variable explains {best_r2*100:.2f}% of variance in sales")

print("\n2. MULTIPLE REGRESSION ADVANTAGE:")
print(f"   - Full model R² = {r2_all:.4f}")
print(f"   - Improvement over single predictor = {improvement:.2f}%")
print(f"   - Shows that other features provide meaningful information")

print("\n3. ADJUSTED R² REVEALS OVERFITTING:")
print(f"   - Full model: R² = {r2_all:.4f}, Adjusted R² = {adj_r2_all:.4f}")
print(f"   - Reduced model: R² = {r2_reduced:.4f}, Adjusted R² = {adj_r2_reduced:.4f}")
print(f"   - Adjusted R² is HIGHER for reduced model!")
print(f"   - This indicates the full model includes unnecessary features")

print("\n4. FEATURE IMPORTANCE:")
print("   Top 5 features by absolute coefficient:")
print(coef_abs.sort_values('Abs_Coefficient', ascending=False).head(5).to_string(index=False))

print("\n5. MODEL SELECTION INSIGHT:")
print("   - Using Adjusted R² as selection criterion leads to simpler, better models")
print("   - Removing low-impact features improves generalization without losing predictive power")
print("   - This is the principle of model parsimony")

print("\n6. PRACTICAL IMPLICATIONS for Car Seat Company:")
print("   - Focus on key factors: Price, CompPrice, Advertising, and ShelveLoc")
print("   - Less important variables can be ignored in business strategy")
print("   - Model is more interpretable with fewer features")

<br>
<br> 
<br>

----